In [28]:
import sys
sys.path.insert(0, '/app/04-evaluation/code')

from ingest import load_faq_data
documents = load_faq_data()


documents_llm = [] 

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)






doc

{'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': "My homework answer doesn't match any of the options",
 'answer': "Common causes, in order of frequency:\n\n1. Wrong column slice or filter — apply filters BEFORE selecting columns / `.head(n)` / `.values`.\n2. Log transform applied where it shouldn't be (or not applied where it should).\n3. Rounding too early — only round the final answer, not intermediate values, unless explicitly told to.\n4. Different sklearn / numpy / Python versions — pin them via `requirements.txt`, `Pipfile.lock`, or `uv.lock`.\n5. Different train/val/test split logic — `train_test_split` shuffles by default; manual `np.random.shuffle` produces a different ordering than sklearn's.\n\nIf after these checks your answer still doesn't match, pick the closest option — the homework explicitly allows it.",
 'doc_id': 'ab183bd688'}

In [29]:
doc = documents[0]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])


from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

9e508f2212
Course: When does the course start?
A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.


In [30]:
import os
print(os.path.exists('/app/04-evaluation/code/ingest.py'))
print(os.listdir('/app/04-evaluation'))

True
['.ipynb_checkpoints', '01-data-gen.ipynb', 'code']


In [31]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [32]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


#openai_client = OpenAI()    

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [33]:
import json

user_prompt = json.dumps(doc)
user_prompt

'{"course": "data-engineering-zoomcamp", "section": "General Course-Related Questions", "question": "Course: When does the course start?", "answer": "A new cohort runs roughly January\\u2013April every year. For the current cohort\'s exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\\n\\n- Register via the link in the course repo before the cohort starts.\\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\\n- Join DataTalks.Club\'s [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.", "doc_id": "9e508f2212"}'

In [34]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [35]:
response = openai_client.responses.parse(
        model="openai/gpt-oss-20b",
        input=messages,
        text_format=Questions

)


In [36]:
response.output_parsed
result = response.output_parsed

print(result)

questions=['When does the new cohort of the Data Engineering Zoomcamp usually begin each year?', 'By what date should I register for the upcoming cohort of the Data Engineering Zoomcamp?', 'Where can I find the official registration link for the Data Engineering Zoomcamp cohort?', 'How can I get announcements and updates about the Data Engineering Zoomcamp as they happen?', 'Which Slack channel should I join to stay connected with the course community?']


In [37]:
print(result.questions[2])

Where can I find the official registration link for the Data Engineering Zoomcamp cohort?


In [38]:
for i in (result.questions):
    print(i)

When does the new cohort of the Data Engineering Zoomcamp usually begin each year?
By what date should I register for the upcoming cohort of the Data Engineering Zoomcamp?
Where can I find the official registration link for the Data Engineering Zoomcamp cohort?
How can I get announcements and updates about the Data Engineering Zoomcamp as they happen?
Which Slack channel should I join to stay connected with the course community?


In [39]:
len(documents)

1377

In [40]:
sys.path.insert(0, '/app/04-evaluation/code')

from evaluation_utils import llm_structured

In [43]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxnyqw8set59syvsm45qgysp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7022, Requested 1569. Please try again in 4.4325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [44]:
from evaluation_utils import calc_price

In [45]:
cost = calc_price(usage)

cost

{'input_cost': 0.000312,
 'output_cost': 0.003186,
 'total_cost': 0.0034980000000000002}

In [46]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': 'When does the new cohort of the Data Engineering Zoomcamp usually begin each year?',
  'document': '9e508f2212'},
 {'question': 'By what date should I register for the upcoming cohort of the Data Engineering Zoomcamp?',
  'document': '9e508f2212'},
 {'question': 'Where can I find the official registration link for the Data Engineering Zoomcamp cohort?',
  'document': '9e508f2212'},
 {'question': 'How can I get announcements and updates about the Data Engineering Zoomcamp as they happen?',
  'document': '9e508f2212'},
 {'question': 'Which Slack channel should I join to stay connected with the course community?',
  'document': '9e508f2212'}]

## 4.3 -- Ground Truth for All Documents

In [47]:
from evaluation_utils import llm_structured_retry

In [48]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

In [49]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kxnyqw8set59syvsm45qgysp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7087, Requested 1569. Please try again in 4.92s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Parallel Processing

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/1377 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)